In [2]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/sst2")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})


In [1]:
from tqdm.auto import tqdm

/Users/farabi/Farabi/ML-Projects/LLM/SST2-Sentiment/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# Convert Hugging Face dataset to a Pandas DataFrame
train_df = dataset["train"].to_pandas()

# Normalize sentences to identify duplicates
groups = (
    train_df["sentence"]
    .str.lower()
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

# Initialize Stratified Group K-Fold
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Select the first fold
train_idx, val_idx = next(
    sgkf.split(
        X=train_df["sentence"],
        y=train_df["label"],
        groups=groups
    )
)

# Create training and validation subsets
internal_train_df = train_df.iloc[train_idx].copy()
internal_val_df = train_df.iloc[val_idx].copy()

print("Training examples:", len(internal_train_df))
print("Validation examples:", len(internal_val_df))

Training examples: 53879
Validation examples: 13470


In [4]:
from datasets import Dataset

train_bert_ds = Dataset.from_pandas(
    internal_train_df[["sentence", "label"]],
    preserve_index=False
)

val_bert_ds = Dataset.from_pandas(
    internal_val_df[["sentence", "label"]],
    preserve_index=False
)

print(train_bert_ds)
print(val_bert_ds)

print("\nFirst training example:")
print(train_bert_ds[0])

Dataset({
    features: ['sentence', 'label'],
    num_rows: 53879
})
Dataset({
    features: ['sentence', 'label'],
    num_rows: 13470
})

First training example:
{'sentence': 'hide new secretions from the parental units ', 'label': 0}


In [5]:
from transformers import AutoTokenizer

# Load the pretrained BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Select one sentence from our training dataset
sentence = train_bert_ds[0]["sentence"]

# Tokenize the sentence
encoded = tokenizer(
    sentence,
    truncation=True,
    max_length=128,
    padding=False
)

print("Original sentence:", sentence)
print("\nTokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("\nInput IDs:", encoded["input_ids"])
print("\nAttention mask:", encoded["attention_mask"])

Original sentence: hide new secretions from the parental units 

Tokens: ['[CLS]', 'hide', 'new', 'secret', '##ions', 'from', 'the', 'parental', 'units', '[SEP]']

Input IDs: [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102]

Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [6]:
# Define our tokenization function
def tokenize_function(batch):
    return tokenizer(
        batch["sentence"],
        truncation=True,
        max_length=128,
        padding=False
    )

# Tokenize the training dataset
train_bert_ds = train_bert_ds.map(
    tokenize_function,
    batched=True
)

# Tokenize the validation dataset
val_bert_ds = val_bert_ds.map(
    tokenize_function,
    batched=True
)

# Examine the result
print(train_bert_ds)
print("\nFirst tokenized example:")
print(train_bert_ds[0])

Map: 100%|██████████| 13470/13470 [00:00<00:00, 138009.62 examples/s]

Dataset({
    features: ['sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 53879
})

First tokenized example:
{'sentence': 'hide new secretions from the parental units ', 'label': 0, 'input_ids': [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [7]:
# Remove the original sentence column
train_bert_ds = train_bert_ds.remove_columns(["sentence"])
val_bert_ds = val_bert_ds.remove_columns(["sentence"])

# Rename label to labels
train_bert_ds = train_bert_ds.rename_column("label", "labels")
val_bert_ds = val_bert_ds.rename_column("label", "labels")

# Examine the updated datasets
print(train_bert_ds)
print(val_bert_ds)

print("\nFirst training example:")
print(train_bert_ds[0])

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 53879
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 13470
})

First training example:
{'labels': 0, 'input_ids': [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [8]:
from transformers import DataCollatorWithPadding

collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt"
)

In [9]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_bert_ds,
    batch_size=16,
    shuffle=True,
    collate_fn=collator
)

val_loader = DataLoader(
    val_bert_ds,
    batch_size=16,
    shuffle=False,
    collate_fn=collator
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 3368
Validation batches: 842


In [10]:
batch = next(iter(train_loader))

for key, value in batch.items():
    print(f"{key}: {value.shape}")

print("\nLabels:")
print(batch["labels"])

print("\nFirst sentence tokens:")
print(tokenizer.convert_ids_to_tokens(batch["input_ids"][0]))

labels: torch.Size([16])
input_ids: torch.Size([16, 33])
token_type_ids: torch.Size([16, 33])
attention_mask: torch.Size([16, 33])

Labels:
tensor([0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1])

First sentence tokens:
['[CLS]', 'flag', '##rant', '##ly', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [11]:
'''
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("Model architecture:", type(model).__name__)
print("Hidden dimension:", model.config.hidden_size)
print("Number of transformer layers:", model.config.num_hidden_layers)
print("Number of attention heads:", model.config.num_attention_heads)

print("\nClassification head:")
print(model.classifier)
'''

'\nfrom transformers import AutoModelForSequenceClassification\n\nmodel = AutoModelForSequenceClassification.from_pretrained(\n    "bert-base-uncased",\n    num_labels=2\n)\n\nprint("Model architecture:", type(model).__name__)\nprint("Hidden dimension:", model.config.hidden_size)\nprint("Number of transformer layers:", model.config.num_hidden_layers)\nprint("Number of attention heads:", model.config.num_attention_heads)\n\nprint("\nClassification head:")\nprint(model.classifier)\n'

In [12]:
'''
import torch

# Set the model to evaluation mode
model.eval()

# Select only 2 sentences from our batch
small_batch = {
    key: value[:2]
    for key, value in batch.items()
}

# Perform a forward pass
with torch.no_grad():
    outputs = model(**small_batch)

# Extract logits and loss
logits = outputs.logits
loss = outputs.loss

# Convert logits into probabilities
probabilities = torch.softmax(logits, dim=-1)

print("Logits shape:", logits.shape)
print("\nLogits:")
print(logits)

print("\nLoss:", loss.item())

print("\nProbabilities:")
print(probabilities)

print("\nPredicted labels:", probabilities.argmax(dim=-1))
print("Actual labels:", small_batch["labels"])
'''

'\nimport torch\n\n# Set the model to evaluation mode\nmodel.eval()\n\n# Select only 2 sentences from our batch\nsmall_batch = {\n    key: value[:2]\n    for key, value in batch.items()\n}\n\n# Perform a forward pass\nwith torch.no_grad():\n    outputs = model(**small_batch)\n\n# Extract logits and loss\nlogits = outputs.logits\nloss = outputs.loss\n\n# Convert logits into probabilities\nprobabilities = torch.softmax(logits, dim=-1)\n\nprint("Logits shape:", logits.shape)\nprint("\nLogits:")\nprint(logits)\n\nprint("\nLoss:", loss.item())\n\nprint("\nProbabilities:")\nprint(probabilities)\n\nprint("\nPredicted labels:", probabilities.argmax(dim=-1))\nprint("Actual labels:", small_batch["labels"])\n'

In [13]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")

elif torch.backends.mps.is_available():
    device = torch.device("mps")

else:
    device = torch.device("cpu")

print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: mps


In [14]:
'''
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

print("Optimizer:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Weight decay:", optimizer.param_groups[0]["weight_decay"])
'''

'\nfrom torch.optim import AdamW\n\noptimizer = AdamW(\n    model.parameters(),\n    lr=2e-5,\n    weight_decay=0.01\n)\n\nprint("Optimizer:", type(optimizer).__name__)\nprint("Learning rate:", optimizer.param_groups[0]["lr"])\nprint("Weight decay:", optimizer.param_groups[0]["weight_decay"])\n'

In [15]:
'''
# Switch BERT to training mode
model.train()

# Take only 2 examples for our first training test
small_batch = {
    key: value[:2].to(device)
    for key, value in batch.items()
}

# Step 1: Clear previous gradients
optimizer.zero_grad()

# Step 2: Forward propagation
outputs = model(**small_batch)

# Step 3: Calculate loss
loss = outputs.loss

print("Training loss:", loss.item())

# Step 4: Backpropagation
loss.backward()

# Examine the calculated gradients
print(
    "Classifier gradient norm:",
    model.classifier.weight.grad.norm().item()
)

# Step 5: Update BERT's parameters
optimizer.step()

print("One training step completed!")
'''

'\n# Switch BERT to training mode\nmodel.train()\n\n# Take only 2 examples for our first training test\nsmall_batch = {\n    key: value[:2].to(device)\n    for key, value in batch.items()\n}\n\n# Step 1: Clear previous gradients\noptimizer.zero_grad()\n\n# Step 2: Forward propagation\noutputs = model(**small_batch)\n\n# Step 3: Calculate loss\nloss = outputs.loss\n\nprint("Training loss:", loss.item())\n\n# Step 4: Backpropagation\nloss.backward()\n\n# Examine the calculated gradients\nprint(\n    "Classifier gradient norm:",\n    model.classifier.weight.grad.norm().item()\n)\n\n# Step 5: Update BERT\'s parameters\noptimizer.step()\n\nprint("One training step completed!")\n'

In [16]:
'''
# Switch BERT to training mode
model.train()

total_loss = 0
num_batches = 3

for step, batch in enumerate(train_loader):

    # Stop after 3 batches for testing
    if step >= num_batches:
        break

    # Use 4 examples per batch for this local test
    batch = {
        key: value[:4].to(device)
        for key, value in batch.items()
    }

    # Clear gradients
    optimizer.zero_grad()

    # Forward propagation
    outputs = model(**batch)

    # Calculate loss
    loss = outputs.loss

    # Backpropagation
    loss.backward()

    # Update parameters
    optimizer.step()

    # Accumulate training loss
    total_loss += loss.item()

    print(f"Batch {step + 1} | Loss: {loss.item():.4f}")

# Calculate average loss
avg_loss = total_loss / num_batches

print(f"\nAverage training loss: {avg_loss:.4f}")
'''

'\n# Switch BERT to training mode\nmodel.train()\n\ntotal_loss = 0\nnum_batches = 3\n\nfor step, batch in enumerate(train_loader):\n\n    # Stop after 3 batches for testing\n    if step >= num_batches:\n        break\n\n    # Use 4 examples per batch for this local test\n    batch = {\n        key: value[:4].to(device)\n        for key, value in batch.items()\n    }\n\n    # Clear gradients\n    optimizer.zero_grad()\n\n    # Forward propagation\n    outputs = model(**batch)\n\n    # Calculate loss\n    loss = outputs.loss\n\n    # Backpropagation\n    loss.backward()\n\n    # Update parameters\n    optimizer.step()\n\n    # Accumulate training loss\n    total_loss += loss.item()\n\n    print(f"Batch {step + 1} | Loss: {loss.item():.4f}")\n\n# Calculate average loss\navg_loss = total_loss / num_batches\n\nprint(f"\nAverage training loss: {avg_loss:.4f}")\n'

In [20]:
def train_one_epoch(model, dataloader, optimizer, device):

    # Enable training mode
    model.train()

    total_loss = 0.0
    total_examples = 0

    for batch in tqdm(dataloader, desc="Training", leave=False):

        # Move batch tensors to the GPU
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward propagation
        outputs = model(**batch)

        # Calculate loss
        loss = outputs.loss

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Calculate batch size
        batch_size = batch["labels"].size(0)

        # Accumulate loss
        total_loss += loss.item() * batch_size
        total_examples += batch_size

    # Average loss per example
    avg_loss = total_loss / total_examples

    return avg_loss

In [21]:
from sklearn.metrics import accuracy_score, f1_score
import torch

def evaluate(model, dataloader, device):

    # Switch to evaluation mode
    model.eval()

    total_loss = 0.0
    total_examples = 0

    all_predictions = []
    all_labels = []

    # Disable gradient calculations
    with torch.no_grad():

        for batch in tqdm(dataloader, desc="Validation", leave=False):

            # Move tensors to the selected device
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            # Forward pass only
            outputs = model(**batch)

            # Get loss and logits
            loss = outputs.loss
            logits = outputs.logits

            # Convert logits into predicted classes
            predictions = torch.argmax(logits, dim=-1)

            # Accumulate validation loss
            batch_size = batch["labels"].size(0)

            total_loss += loss.item() * batch_size
            total_examples += batch_size

            # Store predictions and actual labels
            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(batch["labels"].cpu().tolist())

    # Calculate evaluation metrics
    avg_loss = total_loss / total_examples

    accuracy = accuracy_score(all_labels, all_predictions)

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro"
    )

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }

In [ ]:
from transformers import AutoModelForSequenceClassification
from torch.optim import AdamW
import torch

# Experiment configuration
NUM_EPOCHS = 2
LEARNING_RATE = 2e-5
SAVE_DIR = "/kaggle/working/bert_sst2_best"

# Set random seed
torch.manual_seed(42)

# Start from a fresh pretrained BERT model
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

model = model.to(device)

# Create a fresh optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

# Track the best validation Macro F1
best_f1 = -1.0

# Complete training loop
for epoch in range(NUM_EPOCHS):

    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}", flush=True)

    # Train for one epoch
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        device
    )

    # Evaluate on validation data
    metrics = evaluate(
        model,
        val_loader,
        device
    )

    print(f"Training Loss: {train_loss:.4f}", flush=True)
    print(f"Validation Loss: {metrics['loss']:.4f}", flush=True)
    print(f"Validation Accuracy: {metrics['accuracy']:.4f}", flush=True)
    print(f"Validation Macro F1: {metrics['macro_f1']:.4f}", flush=True)

    # Save the best-performing model
    if metrics["macro_f1"] > best_f1:

        best_f1 = metrics["macro_f1"]

        model.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)

        print(
            f"New best model saved! Macro F1: {best_f1:.4f}",
            flush=True
        )

print("\nTraining completed!", flush=True)
print(f"Best Validation Macro F1: {best_f1:.4f}", flush=True)